# Reproduce Manuscript Figures

This notebook reproduces the three final manuscript figures and verifies the headline results from the compact publication data retained in this repository. Run it from the repository root; the large gridded source products are not required.


In [ ]:
from pathlib import Path
import runpy

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'figure_data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'figure_data').exists():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory.')

DATA = ROOT / 'data'
FIG_DATA = DATA / 'figure_data'
FIG = ROOT / 'figures'
FIG.mkdir(exist_ok=True)

ELEVATION_ORDER = ['<100 m', '100-499 m', '500-1499 m', '1500-2499 m', '2500-3499 m', '>=3500 m']
SETTLEMENT_ORDER = ['Low-density rural', 'Rural cluster', 'Peri-urban', 'Semi-dense urban', 'Dense urban', 'Urban centre']
ELEVATION_LABELS = {'<100 m': '<100 m', '100-499 m': '100–499 m', '500-1499 m': '500–1,499 m', '1500-2499 m': '1,500–2,499 m', '2500-3499 m': '2,500–3,499 m', '>=3500 m': '≥3,500 m'}
AGE_COLORS = {'0-14': '#3B9AB2', '15-64': '#E1AF00', '65+': '#F21A00'}
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
FIGURE_FONT = 'Helvetica' if 'Helvetica' in available_fonts else ('Arial' if 'Arial' in available_fonts else 'DejaVu Sans')
mpl.rcParams.update({'font.family': FIGURE_FONT, 'font.size': 6.6, 'axes.titlesize': 7.4, 'axes.labelsize': 6.8, 'xtick.labelsize': 6.0, 'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.55, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.facecolor': 'white'})

def savefig(name, fig):
    fig.savefig(FIG / f'{name}.pdf', bbox_inches='tight', facecolor='white')

def style_axes(ax, grid_axis='x'):
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_linewidth(0.55)
        spine.set_color('0.25')
    if grid_axis:
        ax.grid(axis=grid_axis, color='0.90', linewidth=0.45)
        ax.set_axisbelow(True)

def panel_label(ax, label):
    ax.text(0.5, 1.115, label, transform=ax.transAxes, fontweight='bold', va='bottom', ha='center', fontsize=8.8)


## Load retained publication data


In [ ]:
main_fig01_data = pd.read_csv(FIG_DATA / 'fig1_elevation_summary.csv')
fig1_age_contribution_values = pd.read_csv(FIG_DATA / 'fig1_age_contributions.csv')
main_fig02_static = pd.read_csv(FIG_DATA / 'fig2_elevation_settlement_age_shares.csv')
regional_fig03 = pd.read_csv(FIG_DATA / 'fig3_region_elevation_growth.csv')


## Essential data checks


In [ ]:
assert list(main_fig01_data['elevation_group']) == ELEVATION_ORDER
assert len(fig1_age_contribution_values) == 18
assert len(main_fig02_static) == 36
assert len(regional_fig03) == 36
assert not any(table.isna().any().any() for table in [main_fig01_data, fig1_age_contribution_values, main_fig02_static, regional_fig03])
assert np.allclose(fig1_age_contribution_values.groupby('elevation_group')['contribution_to_total_growth_pp'].sum(), fig1_age_contribution_values.groupby('elevation_group')['total_growth_percent'].first(), atol=1e-8)
global_pop_2025 = main_fig01_data['population_total'].sum()


## Figure 1

Global population by elevation group, 2025 age composition, 2015–2025 change by broad age class, and the 2025 cumulative elevation profile.


In [ ]:
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
import matplotlib.patheffects as pe

fig1_age_contrib = fig1_age_contribution_values.copy()
contrib_pivot = (
    fig1_age_contrib
    .pivot(index='elevation_group', columns='age_group', values='contribution_to_total_growth_pp')
    .reindex(index=ELEVATION_ORDER, columns=['0-14', '15-64', '65+'])
)
contrib_check = (
    fig1_age_contrib.groupby('elevation_group', as_index=False)
    .agg(sum_contribution_pp=('contribution_to_total_growth_pp', 'sum'), total_growth_percent=('total_growth_percent', 'first'))
)
contrib_check['difference_pp'] = contrib_check['sum_contribution_pp'] - contrib_check['total_growth_percent']
assert np.allclose(contrib_check['difference_pp'], 0, atol=1e-8), 'Figure 1C age contributions do not sum to total growth.'

FIG1_YEAR_COLORS = {'2015': '#C6CDF7', '2025': '#5F7FC8'}
FIG1_AGE_COLORS = {'0-14': '#3B9AB2', '15-64': '#E1AF00', '65+': '#E85D3F'}
FIG1_PROFILE_LINE_COLOR = '#111111'
FIG1_GRID_COLOR = '0.90'
FIG1_AXIS_COLOR = '0.25'
FIG1_SOURCE_PROFILE_DATA = FIG_DATA / 'fig1_elevation_profile_2025.csv'
FIG1_PROFILE_YEAR = 2025
FIG1_MIN_ELEV_M = 0
FIG1_MAX_ELEV_M = 8000
FIG1_PROFILE_YMAX_M = 5000
FIG1_SYMLINTHRESH_M = 10
FIG1_PROFILE_Y_LABEL_TICKS_M = [0, 148, 500, 1500, 3500]
FIG1_AGE_PROFILE_GROUPS = [
    ('0-14', ['young_0_14']),
    ('15-64', ['working_age_15_64']),
    ('65+', ['old_age_65_plus', 'older_65_plus']),
]
FIG1_FIGURE_WIDTH_IN = 7.25
FIG1_FIGURE_HEIGHT_IN = 7.60
FIG1_PANEL_WIDTH_IN = 2.65
FIG1_PANEL_HEIGHT_IN = 2.35
FIG1_LEFT_IN = 0.92
FIG1_COLUMN_GAP_IN = 0.70
FIG1_BOTTOM_ROW_BOTTOM_IN = 1.10
FIG1_TOP_ROW_BOTTOM_IN = 4.45
FIG1_TITLE_SIZE = 9.4
FIG1_LABEL_SIZE = 8.7
FIG1_TICK_SIZE = 7.8
FIG1_PANEL_LABEL_SIZE = 11.0
FIG1_LEGEND_TEXT_SIZE = 7.6
FIG1_ANNOTATION_SIZE = 7.4
FIG1_TOTAL_CHANGE_TEXT_SIZE = 8.8
FIG1_PROFILE_LEGEND_SIZE = 8.0
FIG1_MEDIAN_TEXT_SIZE = 8.2


def fig1_fmt_elevation(x, _pos=None):
    return f'{x:,.0f}'


def fig1_build_cdf(df):
    cdf = (
        df.groupby('elevation_threshold_m', as_index=False)['population_count']
        .sum()
        .rename(columns={'elevation_threshold_m': 'elevation_m'})
        .sort_values('elevation_m')
    )
    all_thresholds = pd.DataFrame({'elevation_m': np.arange(FIG1_MIN_ELEV_M, FIG1_MAX_ELEV_M + 1)})
    cdf = all_thresholds.merge(cdf, on='elevation_m', how='left').fillna({'population_count': 0.0})
    cdf['cum_pop'] = cdf['population_count'].cumsum()
    total_pop = float(cdf['population_count'].sum())
    cdf['cum_frac_percent'] = cdf['cum_pop'] / total_pop * 100
    return cdf


def fig1_load_global_profile():
    if not FIG1_SOURCE_PROFILE_DATA.exists():
        raise FileNotFoundError(f'Missing profile source data: {FIG1_SOURCE_PROFILE_DATA.relative_to(ROOT)}')
    df = pd.read_csv(FIG1_SOURCE_PROFILE_DATA)
    df['elevation_m'] = pd.to_numeric(df['elevation_m'], errors='coerce')
    df['population_count'] = pd.to_numeric(df['population_count'], errors='coerce').fillna(0)
    df = df.dropna(subset=['elevation_m']).copy()
    df['elevation_threshold_m'] = df['elevation_m'].clip(lower=FIG1_MIN_ELEV_M, upper=FIG1_MAX_ELEV_M).round().astype(int)

    cdf = fig1_build_cdf(df)
    age_cdfs = {
        label: fig1_build_cdf(df[df['broad_age_group'].isin(source_groups)])
        for label, source_groups in FIG1_AGE_PROFILE_GROUPS
    }
    total_pop = float(cdf['population_count'].sum())
    median_elevation = float(cdf.loc[cdf['cum_pop'].ge(total_pop / 2), 'elevation_m'].iloc[0])
    return cdf, age_cdfs, median_elevation


def legend_text_color(hex_color, label=None):
    if label in {'2025', '0-14', '65+'}:
        return 'white'
    rgb = mpl.colors.to_rgb(hex_color)
    luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
    return 'white' if luminance < 0.52 else '0.12'


def draw_box_legend(fig, *, items, center_x, y0, box_width=0.074, box_height=0.044, gap=0.010):
    total_width = len(items) * box_width + (len(items) - 1) * gap
    x0 = center_x - total_width / 2
    for i, (label, color) in enumerate(items):
        x = x0 + i * (box_width + gap)
        rect = mpl.patches.Rectangle(
            (x, y0),
            box_width,
            box_height,
            transform=fig.transFigure,
            facecolor=color,
            edgecolor='white',
            linewidth=0.55,
            clip_on=False,
        )
        fig.add_artist(rect)
        fig.text(
            x + box_width / 2,
            y0 + box_height / 2,
            label,
            ha='center',
            va='center',
            fontsize=FIG1_LEGEND_TEXT_SIZE,
            color=legend_text_color(color, label),
            fontweight='bold' if label in {'2025', '0-14', '65+'} else 'normal',
        )


def draw_stacked_box_legend(ax, *, items, box_width=0.24, box_height=0.095, gap=0.018, inset=0.025):
    x0 = 1 - inset - box_width
    for i, (label, color) in enumerate(items):
        y0 = 1 - inset - box_height - i * (box_height + gap)
        rect = mpl.patches.Rectangle(
            (x0, y0),
            box_width,
            box_height,
            transform=ax.transAxes,
            facecolor=color,
            edgecolor='white',
            linewidth=0.55,
            clip_on=False,
            zorder=10,
        )
        ax.add_patch(rect)
        ax.text(
            x0 + box_width / 2,
            y0 + box_height / 2,
            label,
            transform=ax.transAxes,
            ha='center',
            va='center',
            fontsize=FIG1_LEGEND_TEXT_SIZE,
            color=legend_text_color(color, label),
            fontweight='bold' if label == '2025' else 'normal',
            zorder=11,
        )


def panel_label_centered(ax, label):
    ax.text(
        0.5,
        1.1,
        label,
        transform=ax.transAxes,
        fontweight='bold',
        va='bottom',
        ha='center',
        fontsize=FIG1_PANEL_LABEL_SIZE,
    )


def add_axis_by_inches(fig, *, left, bottom, width, height):
    fig_width, fig_height = fig.get_size_inches()
    return fig.add_axes([left / fig_width, bottom / fig_height, width / fig_width, height / fig_height])


def draw_population_panel(ax, fig, y, labels):
    h = 0.34
    xmin = 0.008
    ax.barh(
        y - h / 2,
        main_fig01_data['population_2015_billions'] - xmin,
        left=xmin,
        height=h,
        color=FIG1_YEAR_COLORS['2015'],
        edgecolor='white',
        linewidth=0.45,
    )
    ax.barh(
        y + h / 2,
        main_fig01_data['population_2025_billions'] - xmin,
        left=xmin,
        height=h,
        color=FIG1_YEAR_COLORS['2025'],
        edgecolor='white',
        linewidth=0.45,
    )
    ax.set_xscale('log')
    ax.set_xlim(xmin, 5.0)
    ax.set_xticks([0.01, 0.1, 1, 5])
    ax.set_xticklabels(['10M', '100M', '1B', '5B'])
    ax.set_yticks(y, labels)
    ax.set_xlabel('Population, log scale', fontsize=FIG1_LABEL_SIZE)
    ax.set_ylabel('Elevation group', fontsize=FIG1_LABEL_SIZE)
    ax.set_title('Population by elevation', fontsize=FIG1_TITLE_SIZE)
    ax.tick_params(axis='both', labelsize=FIG1_TICK_SIZE)
    style_axes(ax, grid_axis='x')
    panel_label_centered(ax, 'A')
    draw_stacked_box_legend(
        ax,
        items=[('2015', FIG1_YEAR_COLORS['2015']), ('2025', FIG1_YEAR_COLORS['2025'])],
    )


def draw_age_composition_panel(ax, y, labels):
    left = np.zeros(len(main_fig01_data))
    for col, lab in [('youth_share_percent', '0-14'), ('working_age_share_percent', '15-64'), ('old_age_share_percent', '65+')]:
        vals = main_fig01_data[col].to_numpy()
        ax.barh(y, vals, left=left, color=FIG1_AGE_COLORS[lab], edgecolor='white', linewidth=0.45)
        left += vals
    ax.set_xlim(0, 100)
    ax.set_yticks(y, labels)
    ax.set_ylabel('Elevation group', fontsize=FIG1_LABEL_SIZE)
    ax.set_xlabel('Population share (%)', fontsize=FIG1_LABEL_SIZE)
    ax.set_title('Age composition, 2025', fontsize=FIG1_TITLE_SIZE)
    ax.tick_params(axis='both', labelsize=FIG1_TICK_SIZE)
    style_axes(ax, grid_axis='x')
    panel_label_centered(ax, 'C')


def draw_growth_panel(ax, fig, y, legend_center_x):
    pos_left = np.zeros(len(contrib_pivot))
    neg_left = np.zeros(len(contrib_pivot))
    for age in ['0-14', '15-64', '65+']:
        vals = contrib_pivot[age].to_numpy()
        left = np.where(vals >= 0, pos_left, neg_left)
        ax.barh(y, vals, left=left, color=FIG1_AGE_COLORS[age], edgecolor='white', linewidth=0.45)
        pos_left += np.where(vals > 0, vals, 0)
        neg_left += np.where(vals < 0, vals, 0)
    ax.axvline(0, color='0.28', linewidth=0.7)
    growth_label_x = 20.1
    ax.set_xlim(min(-2, np.nanmin(neg_left) - 0.5), 25.0)
    ax.set_yticks(y)
    ax.tick_params(axis='y', labelleft=False)
    ax.text(
        growth_label_x,
        0.975,
        'Total change',
        transform=ax.get_xaxis_transform(),
        ha='center',
        va='top',
        fontsize=FIG1_TOTAL_CHANGE_TEXT_SIZE,
        fontweight='bold',
        color='0.12',
        clip_on=False,
        path_effects=[pe.withStroke(linewidth=1.1, foreground='white')],
    )
    for i, elev in enumerate(contrib_pivot.index):
        total_growth = fig1_age_contrib.loc[fig1_age_contrib['elevation_group'].eq(elev), 'total_growth_percent'].iloc[0]
        ax.text(
            growth_label_x,
            y[i],
            f'{total_growth:+.1f}%',
            va='center',
            ha='center',
            fontsize=FIG1_TOTAL_CHANGE_TEXT_SIZE,
            color='0.12',
            path_effects=[pe.withStroke(linewidth=1.2, foreground='white')],
        )
    ax.set_xlabel('Contribution to total change\n(pct. points)', fontsize=FIG1_LABEL_SIZE)
    ax.set_title('Population change, 2015–2025', fontsize=FIG1_TITLE_SIZE, linespacing=0.95)
    ax.tick_params(axis='both', labelsize=FIG1_TICK_SIZE)
    style_axes(ax, grid_axis='x')
    panel_label_centered(ax, 'D')
    draw_box_legend(
        fig,
        items=[(a, FIG1_AGE_COLORS[a]) for a in ['0-14', '15-64', '65+']],
        center_x=legend_center_x,
        y0=0.035,
    )


def draw_profile_panel(ax, cdf, age_cdfs, median_elevation):
    median_x = 50.0
    ax.set_xlim(0, 100)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_ylim(FIG1_MIN_ELEV_M, FIG1_PROFILE_YMAX_M)
    ax.set_yscale('symlog', linthresh=FIG1_SYMLINTHRESH_M)
    ax.yaxis.set_major_locator(FixedLocator(FIG1_PROFILE_Y_LABEL_TICKS_M))
    ax.yaxis.set_major_formatter(FuncFormatter(fig1_fmt_elevation))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.grid(True, which='major', axis='x', linestyle='-', linewidth=0.45, color=FIG1_GRID_COLOR, zorder=1)
    for guide_m in FIG1_PROFILE_Y_LABEL_TICKS_M:
        ax.axhline(guide_m, color=FIG1_GRID_COLOR, linewidth=0.45, zorder=1)
    for age_label, age_cdf in age_cdfs.items():
        ax.plot(
            age_cdf['cum_frac_percent'],
            age_cdf['elevation_m'],
            color=FIG1_AGE_COLORS[age_label],
            linewidth=1.05,
            alpha=0.72,
            label=age_label,
            zorder=3,
        )
    ax.plot(
        cdf['cum_frac_percent'],
        cdf['elevation_m'],
        color=FIG1_PROFILE_LINE_COLOR,
        linewidth=1.9,
        alpha=1.0,
        label='Global',
        zorder=5,
    )
    guide_style = (0, (2.4, 2.0))
    age_medians = {
        age_label: float(age_cdf.loc[age_cdf['cum_frac_percent'].ge(50), 'elevation_m'].iloc[0])
        for age_label, age_cdf in age_cdfs.items()
    }
    for age_label, age_median in age_medians.items():
        ax.plot(
            [0, median_x],
            [age_median, age_median],
            color=FIG1_AGE_COLORS[age_label],
            linewidth=0.75,
            alpha=0.82,
            linestyle=guide_style,
            zorder=4,
        )
    ax.plot([0, median_x], [median_elevation, median_elevation], color='0.18', linewidth=0.7, alpha=0.62, linestyle=guide_style, zorder=6)
    ax.plot([median_x, median_x], [FIG1_MIN_ELEV_M, median_elevation], color='0.18', linewidth=0.7, alpha=0.62, linestyle=guide_style, zorder=6)
    ax.text(
        median_x + 2.5,
        30,
        f'50% of the global\npopulation lives below\n{median_elevation:.0f} m',
        ha='left',
        va='center',
        fontsize=FIG1_PROFILE_LEGEND_SIZE,
        fontweight='normal',
        linespacing=1.15,
        color='black',
        zorder=8,
    )
    handles, legend_labels = ax.get_legend_handles_labels()
    order = [legend_labels.index(label) for label in ['Global', '0-14', '15-64', '65+'] if label in legend_labels]
    legend = ax.legend(
        [handles[i] for i in order],
        [legend_labels[i] for i in order],
        loc='lower right',
        fontsize=FIG1_PROFILE_LEGEND_SIZE,
        frameon=True,
        framealpha=0.92,
        edgecolor='0.82',
        handlelength=1.5,
        borderpad=0.28,
        labelspacing=0.20,
    )
    legend.get_frame().set_linewidth(0.45)
    for line in legend.get_lines():
        line.set_linewidth(1.35)
    ax.set_xlabel('Cumulative population\nbelow threshold (%)', fontsize=FIG1_LABEL_SIZE)
    ax.set_ylabel('Elevation threshold (m)', fontsize=FIG1_LABEL_SIZE)
    ax.set_title('Elevation profile', fontsize=FIG1_TITLE_SIZE)
    ax.tick_params(axis='both', labelsize=FIG1_TICK_SIZE)
    style_axes(ax, grid_axis=None)
    panel_label_centered(ax, 'B')


cdf, age_cdfs, median_elevation = fig1_load_global_profile()
y = np.arange(len(main_fig01_data))
labels = [ELEVATION_LABELS[e] for e in main_fig01_data['elevation_group'].astype(str)]

fig = plt.figure(figsize=(FIG1_FIGURE_WIDTH_IN, FIG1_FIGURE_HEIGHT_IN))
ax_population = add_axis_by_inches(
    fig,
    left=FIG1_LEFT_IN,
    bottom=FIG1_TOP_ROW_BOTTOM_IN,
    width=FIG1_PANEL_WIDTH_IN,
    height=FIG1_PANEL_HEIGHT_IN,
)
ax_profile = add_axis_by_inches(
    fig,
    left=FIG1_LEFT_IN + FIG1_PANEL_WIDTH_IN + FIG1_COLUMN_GAP_IN,
    bottom=FIG1_TOP_ROW_BOTTOM_IN,
    width=FIG1_PANEL_WIDTH_IN,
    height=FIG1_PANEL_HEIGHT_IN,
)
ax_age = add_axis_by_inches(
    fig,
    left=FIG1_LEFT_IN,
    bottom=FIG1_BOTTOM_ROW_BOTTOM_IN,
    width=FIG1_PANEL_WIDTH_IN,
    height=FIG1_PANEL_HEIGHT_IN,
)
ax_growth = add_axis_by_inches(
    fig,
    left=FIG1_LEFT_IN + FIG1_PANEL_WIDTH_IN + FIG1_COLUMN_GAP_IN,
    bottom=FIG1_BOTTOM_ROW_BOTTOM_IN,
    width=FIG1_PANEL_WIDTH_IN,
    height=FIG1_PANEL_HEIGHT_IN,
)

draw_population_panel(ax_population, fig, y, labels)
draw_age_composition_panel(ax_age, y, labels)
age_growth_legend_center = (ax_age.get_position().x0 + ax_growth.get_position().x1) / 2
draw_growth_panel(ax_growth, fig, y, age_growth_legend_center)
draw_profile_panel(ax_profile, cdf, age_cdfs, median_elevation)

savefig('fig1', fig)
plt.close(fig)

print(f'Saved figures/fig1.pdf; global median elevation = {median_elevation:.0f} m')


## Figure 2


In [ ]:
runpy.run_path(str(ROOT / 'processing' / 'plot_fig2.py'), run_name='__main__')


## Figure 3

Global elevation bands and continent-by-elevation matrices for 2025 population and 2015–2025 population change.


In [ ]:
runpy.run_path(str(ROOT / 'processing' / 'plot_fig3.py'), run_name='__main__')


## Headline manuscript results


In [ ]:
headline = pd.read_csv(FIG_DATA / 'headline_manuscript_values.csv')
growth = pd.read_csv(DATA / 'country_comparisons' / 'within_country_growth.csv')
age = pd.read_csv(DATA / 'country_comparisons' / 'within_country_age_structure_2025.csv')
assert growth['iso3'].nunique() == len(growth) == 48
assert age['iso3'].nunique() == len(age) == 48
assert set(growth['iso3']) == set(age['iso3'])
assert (growth[['lowland_population_2025', 'highland_population_2025']] >= 100_000).all().all()
for zone in ['lowland', 'highland']:
    assert np.allclose(age[f'{zone}_youth_share_2025_percent'], 100 * age[f'{zone}_youth_population_2025'] / age[f'{zone}_population_2025'])
    assert np.allclose(age[f'{zone}_older_age_share_2025_percent'], 100 * age[f'{zone}_older_population_2025'] / age[f'{zone}_population_2025'])
age_youth = age['highland_minus_lowland_youth_share_difference_pp']
age_older = age['highland_minus_lowland_older_age_share_difference_pp']
growth_difference = growth['highland_minus_lowland_growth_difference_pp']
print(f'Headline value table: {len(headline)} retained manuscript statistics')
print(f'Within-country growth: {(growth_difference > 0).sum()}/48 countries; mean {growth_difference.mean():+.2f} pp; median {growth_difference.median():+.2f} pp')
print(f'Within-country youth share: {(age_youth > 0).sum()}/48 countries; mean {age_youth.mean():+.2f} pp; median {age_youth.median():+.2f} pp')
print(f'Within-country older-age share: {(age_older > 0).sum()}/48 countries; mean {age_older.mean():+.2f} pp; median {age_older.median():+.2f} pp')


## Sensitivity results

The retained settlement-class comparison and robustness summaries are in `data/sensitivity/`. These tables are analysis outputs, not additional manuscript figures.
